In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from config import SIM_CONFIG
from users import generate_users
from sessions import generate_sessions
from events import generate_events
from orders import generate_orders
from order_items import generate_order_items

rng = np.random.default_rng(seed=42)
print("modules loaded successfully")

modules loaded successfully


In [2]:
def _prepare_purchases(df_events, df_sessions, df_users) -> dict:
    """Orchestration adapter: prepares flat purchase arrays for generate_orders."""
    working_df = df_events[df_events["event_name"] == "purchase"].copy()

    working_df = working_df.merge(
        df_sessions[["session_id", "user_id"]], on="session_id", how="left"
    )

    working_df = working_df.merge(
        df_users[["user_id", "latent_income_score"]], on="user_id", how="left"
    )

    return {
        "session_id": working_df["session_id"].to_numpy(),
        "user_id": working_df["user_id"].to_numpy(),
        "event_timestamp": working_df["event_timestamp"].to_numpy(),
        "latent_income_score": working_df["latent_income_score"].to_numpy(),
    }


def _prepare_order_items(df_orders, df_users) -> dict:
    """Orchestration adapter: prepares flat arrays for generate_order_items."""
    working_df = df_orders.merge(
        df_users[["user_id", "latent_income_score"]], on="user_id", how="left"
    )

    return {
        "order_id": working_df["order_id"].to_numpy(),
        "latent_income_score": working_df["latent_income_score"].to_numpy(),
    }


def _prepare_events(df_sessions, df_users) -> dict:
    """Orchestration adapter: prepares flat arrays for generate_events."""
    working_df = df_sessions.merge(
        df_users[["user_id", "latent_digital_literacy", "latent_trust_in_platform"]],
        on="user_id",
        how="left",
    )

    return {
        "session_id": working_df["session_id"].to_numpy(),
        "device_operating_system": working_df["device_operating_system"].to_numpy(),
        "device_group": working_df["device_group"].to_numpy(),
        "latent_digital_literacy": working_df["latent_digital_literacy"].to_numpy(),
        "latent_trust_in_platform": working_df["latent_trust_in_platform"].to_numpy(),
        "session_start_time": working_df["session_start_time"].to_numpy(),
        "session_duration_seconds": working_df["session_duration_seconds"].to_numpy(),
    }

In [3]:
master_ss = np.random.SeedSequence(SIM_CONFIG["random_seed"])
stage_seeds = master_ss.spawn(5)
rng_users = np.random.default_rng(stage_seeds[0])
rng_sessions = np.random.default_rng(stage_seeds[1])
rng_events = np.random.default_rng(stage_seeds[2])
rng_orders = np.random.default_rng(stage_seeds[3])
rng_order_items = np.random.default_rng(stage_seeds[4])

n_workers = SIM_CONFIG["n_workers"]

# test_users_df = generate_users(SIM_CONFIG["target_users"], rng_users)
df_test_users = generate_users(10000, rng_users)
df_test_sessions = generate_sessions(df_test_users, rng_sessions)

session_data = _prepare_events(df_test_sessions, df_test_users)
df_test_events = generate_events(session_data, rng_events, n_workers)

purchase_data = _prepare_purchases(df_test_events, df_test_sessions, df_test_users)
df_test_orders = generate_orders(purchase_data, rng_orders)

order_data = _prepare_order_items(df_test_orders, df_test_users)
df_test_order_items = generate_order_items(order_data, rng_order_items)

print("all tables generated successfully in memory")

print("\nEXPORTING TO CSV")
current_script_dir = os.path.abspath(os.getcwd())
output_dir = os.path.abspath(os.path.join(current_script_dir, '..', 'test_data'))
os.makedirs(output_dir, exist_ok=True)

df_clean_users = df_test_users.drop(columns=[
    "latent_income_score",
    "latent_digital_literacy",
    "latent_trust_in_platform",
])

df_sorted_clean_users = df_clean_users.sort_values(by="account_created_at", ascending=True).reset_index(drop=True)
df_sorted_sessions = df_test_sessions.sort_values(by="session_start_time", ascending=True).reset_index(drop=True)
df_sorted_events = df_test_events.sort_values(by="event_timestamp").reset_index(drop=True)
df_sorted_orders = df_test_orders.sort_values(by="order_timestamp").reset_index(drop=True)

df_sorted_clean_users.to_csv(os.path.join(output_dir, 'users.csv'), index=False)
df_sorted_sessions.to_csv(os.path.join(output_dir, 'sessions.csv'), index=False)
df_sorted_events.to_csv(os.path.join(output_dir, 'events.csv'), index=False)
df_sorted_orders.to_csv(os.path.join(output_dir, 'orders.csv'), index=False)
df_test_order_items.to_csv(os.path.join(output_dir, 'order_items.csv'), index=False)

print(f"export completed in {output_dir}")

Generating 10000 users
users table generated!
Generating 81796 sessions
sessions table generated!
Generating events for 81796 sessions
Generating 2518 orders for purchases events
orders table generated!
Generating order items for 2518 orders
order items table generated!
all tables generated successfully in memory

EXPORTING TO CSV
export completed in /home/ruicchi/github-projects/ph-ecommerce-data-simulator/test_data
